# Scan over layers with jax.checkpoint

Deep models in JAX usually stack their layers' weights along a leading axis and run the layers with `jax.lax.scan`, which compiles one layer body instead of $L$. To cut activation memory, the body is wrapped in `jax.checkpoint` (also called `jax.remat`): the backward pass recomputes each layer's internals instead of storing them.

Each of the $L$ residual MLP blocks maps `(B, D)` to `(B, D)`:

$$h \leftarrow h + \mathrm{gelu}(h W_1 + b_1)\, W_2 + b_2$$

with the **tanh approximation** of GELU, `jax.nn.gelu(..., approximate=True)`:

$$\mathrm{gelu}(z) = \tfrac12 z \left(1 + \tanh\!\left(\sqrt{2/\pi}\,(z + 0.044715 z^3)\right)\right)$$

Write `deep_mlp_loss_and_grads(params, x, y, remat)`:

- `params`: `{'w1': (L, D, F), 'b1': (L, F), 'w2': (L, F, D), 'b2': (L, D)}`; `x`, `y`: `(B, D)`.
- Loss: $\mathrm{mean}\bigl((h_L - y)^2\bigr)$ over all elements, where $h_0 = x$.
- `remat`: Python string.
  - `'none'`: plain scan;
  - `'full'`: wrap the block in `jax.checkpoint` (save only the block input);
  - `'dots'`: wrap it in `jax.checkpoint` with `policy=jax.checkpoint_policies.dots_with_no_batch_dims_saveable` (keep matmul outputs, recompute the cheap elementwise ops).

Return `(loss, grads)`: a 0-d array and a dict shaped like `params`. float32. All three settings must give the same numbers, since rematerialisation changes memory, not math. The tests check values only; use `jax.ad_checkpoint.print_saved_residuals` in Colab to see what each policy keeps.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def deep_mlp_loss_and_grads(params, x, y, remat):
    """Stacked params with a leading layer axis. Return (loss, grads dict)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__dict__": [["w1", {"__nd__": [[[1.0]]], "dtype": "float64"}], ["b1", {"__nd__": [[0.0]], "dtype": "float64"}], ["w2", {"__nd__": [[[1.0]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.0]], "dtype": "float64"}]]}, {"__nd__": [[1.0]], "dtype": "float64"}, {"__nd__": [[0.0]], "dtype": "float64"}, "none"], "e": {"__tup__": [{"__nd__": 3.3899879455566406, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [[3.987889528274536]], "dtype": "float32"}], ["b2", {"__nd__": [[3.6823840141296387]], "dtype": "float32"}], ["w1", {"__nd__": [[[3.987889528274536]]], "dtype": "float32"}], ["w2", {"__nd__": [[[3.0975921154022217]]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[1.0]]], "dtype": "float64"}], ["b1", {"__nd__": [[0.0]], "dtype": "float64"}], ["w2", {"__nd__": [[[1.0]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.0]], "dtype": "float64"}]]}, {"__nd__": [[1.0]], "dtype": "float64"}, {"__nd__": [[0.0]], "dtype": "float64"}, "full"], "e": {"__tup__": [{"__nd__": 3.3899879455566406, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [[3.987889528274536]], "dtype": "float32"}], ["b2", {"__nd__": [[3.6823840141296387]], "dtype": "float32"}], ["w1", {"__nd__": [[[3.987889528274536]]], "dtype": "float32"}], ["w2", {"__nd__": [[[3.0975921154022217]]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[-0.4192855274112129, -0.3612028470286608, -0.3906807018168935, -0.18680561011316651, 0.43418403461809163], [-0.48039416748463415, 0.28319083928007344, 0.3787415747147886, -0.10663818771331546, -0.2102808260027422], [0.14163438620723923, 0.3643672995613227, 0.6466593835203878, 0.26754344782969486, 0.4781527997235746]], [[-0.13508747010156097, -0.2622650951163073, -0.5159319761748147, -0.3461107760237983, -0.10903011495481445], [0.057898416625246404, -0.3939301728367635, 0.24482575917924335, -0.03091468186972795, -0.19266032253905163], [-0.22958662247510914, 0.6062064818649023, 0.05076850478631665, -0.26937796177887346, -0.2861946897121853]], [[0.551347329140022, -0.17350526896299, 0.35661122755970465, 0.42698216435196895, -0.6973204069037421], [0.4052924680488905, -0.06252963971331643, -0.3252385764805109, -0.1631527343977792, 0.16826873072021295], [0.11219296660863498, 0.12765424116862145, -0.21746880496068602, 0.3837251464090552, 0.4506391436689833]], [[0.5502191430578932, 0.3001387933393551, 0.5682943638621228, 0.6281780863270451, 0.21127679831450835], [0.26434863384995794, 0.17466273500794519, 0.1654273206327644, 0.2589733616188232, -0.5126945704582769], [-0.09713951061880531, 0.14635702547392063, -0.6321993134597211, -0.43255216480699327, 0.4753494857062574]]], "dtype": "float64"}], ["b1", {"__nd__": [[-0.10591626265092143, -0.04643473082572405, 0.03751191495613689, -0.0475981543195611, -0.08607988248871355], [-0.05160695588903406, -0.04281975203120115, -0.0579808127566781, 0.018236412585290355, 0.16098145140354342], [0.03563047678382521, 0.1624806827941641, 0.07396646728405731, 0.08360877774221787, 0.05866629023229206], [-0.029890902715660035, 0.0421640044301706, 0.03029302760458512, -0.11123286864271714, 0.07482550537168686]], "dtype": "float64"}], ["w2", {"__nd__": [[[0.1733153347832059, 0.11054851525337535, -0.16753957572157546], [-0.7900152868593612, 0.44023816182313735, -0.021090387958547233], [-0.6532246087821586, 0.1949076543410682, -0.20981926555400907], [0.20607102592511914, -0.030557954511593786, 0.340283461544743], [0.07002978681408285, 0.0838543408157203, 0.04498280736524818]], [[0.3691848138045608, -0.38877660082699256, 0.3530650690235232], [0.2066420052846708, -0.16632183814555473, -0.0007007891451986621], [-0.13395184925102563, 0.017805130180820188, 0.2821009187291959], [-0.20379074090280977, 0.22294793083390066, 0.2822077304036695], [0.19117626293291218, 0.4688649094941081, -0.021591698001178817]], [[0.20785705126439438, 0.7435578486101468, 0.00855960529773087], [-0.0417511812048884, 0.0013083757988210454, 0.1759934322881588], [-0.18546216307214475, -0.07353676770510942, -0.5264478954601624], [-0.3148436907221949, -0.3063361998005162, -0.348020682808397], [-0.1332465615365556, 0.20804015042007737, -0.1149555546005673]], [[-0.39696000106322665, 0.0726865535354096, 0.050852544771356825], [-0.4205193423685934, 0.04452587290877224, -0.32201167898642946], [-0.18973758192025633, -0.22133853823048003, 0.24606697546374598], [-0.24782205085406095, -0.09645216299519556, 0.05427830801339515], [0.30938681884505714, 0.3541006195358496, -0.3061710166364671]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.11834670515455736, 0.029742479238392308, 0.11941634398527437], [0.1052469845112465, -0.05330355135671144, 0.0038009620695226513], [-0.029476355634257802, 0.03730640789512084, 0.16532546900049772], [0.005444305932063776, -0.06639771269713039, 0.2533685004698796]], "dtype": "float64"}]]}, {"__nd__": [[-0.9941497685659371, 0.0007545597703792161, -0.014751124658677827], [-1.270336416589952, 0.7919071077140544, 1.0629464836528513], [-0.08175841953991256, 0.05747763942419942, -0.16287876735084655], [-1.7613771573612063, 0.2435098883321344, -1.243281001073121], [-0.9567565042981796, 0.3968918326546037, 0.9249180794046787], [-0.8669158826508802, 1.140159494666554, 1.7667422643376332]], "dtype": "float64"}, {"__nd__": [[0.17167969525236756, -0.075574596136044, 0.20367313281838842], [-0.3998259722154415, -0.8517494624499372, 0.646338292231823], [0.7208530221284463, -0.49751122444937135, -0.40118783306815836], [-0.6189057971264504, -0.6970104276971278, -0.9638897212512467], [-1.5014293030052674, 0.9053295359389235, -0.3470388429725984], [0.07084728905744735, 1.7912132147949598, -0.4821800858142817]], "dtype": "float64"}, "none"], "e": {"__tup__": [{"__nd__": 2.7083706855773926, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [[-0.17856240272521973, 1.5553213357925415, 1.0138877630233765, -0.07323996722698212, -0.005594316869974136], [-0.24064354598522186, -0.2902124524116516, 0.4828817844390869, 0.6110758781433105, 0.07186605036258698], [0.0298761073499918, 0.1942461133003235, 0.003975574858486652, -0.02646917849779129, 0.17292815446853638], [-0.022871771827340126, 0.049589209258556366, -0.00019163032993674278, -0.005808226298540831, -0.03664916008710861]], "dtype": "float32"}], ["b2", {"__nd__": [[-1.5840599536895752, 0.887549102306366, 0.7382519245147705], [-1.2242109775543213, 0.7016735672950745, 1.0195949077606201], [-1.0765098333358765, 0.669587254524231, 0.9245399236679077], [-1.0673084259033203, 0.6497179865837097, 0.9298481345176697]], "dtype": "float32"}], ["w1", {"__nd__": [[[0.19407159090042114, -1.7063817977905273, -1.1038379669189453, 0.09337353706359863, 0.006486077792942524], [-0.11453713476657867, 1.1506328582763672, 0.7575702667236328, -0.058085549622774124, -0.008597932755947113], [-0.1420070081949234, 1.4735954999923706, 1.0035014152526855, -0.060932550579309464, -0.013011706061661243]], [[0.5117800235748291, 0.6940218210220337, -1.1102592945098877, -1.3619694709777832, -0.09992129355669022], [-0.2449558526277542, -0.40766027569770813, 0.6307113766670227, 0.7373515367507935, 0.027657391503453255], [-0.06572357565164566, -0.2764820158481598, 0.3922184109687805, 0.38863295316696167, -0.04684901610016823]], [[0.009637018665671349, -0.4535869061946869, -0.05718676745891571, 0.01850547268986702, -0.3869594633579254], [-0.0030547287315130234, 0.25099241733551025, 0.030298756435513496, -0.012223544530570507, 0.1903073638677597], [-0.008823418989777565, 0.28170663118362427, 0.0458243153989315, -0.020338552072644234, 0.16013500094413757]], [[0.1442510187625885, -0.12846414744853973, 0.026469485834240913, 0.03421185910701752, 0.019585315138101578], [-0.09679268300533295, 0.08466088771820068, -0.014316953718662262, -0.01993240788578987, -0.00011891928443219513], [-0.08188003301620483, 0.06239943578839302, -0.016017530113458633, -0.01773044653236866, -0.00976521521806717]]], "dtype": "float32"}], ["w2", {"__nd__": [[[-0.13561083376407623, 0.08899946510791779, 0.05589160695672035], [-1.1259511709213257, 0.5256670117378235, 0.5109240412712097], [-1.8674347400665283, 0.8500869870185852, 0.8615576028823853], [-0.31977832317352295, 0.14216358959674835, 0.14908580482006073], [0.07334066182374954, -0.07131727784872055, -0.02076495625078678]], [[-0.11396612972021103, 0.08760681748390198, 0.08220042288303375], [-0.4979013502597809, 0.17605507373809814, 0.4297879636287689], [-1.669232726097107, 0.7760934233665466, 1.3490734100341797], [-0.4987509846687317, 0.3112529516220093, 0.3842129707336426], [-0.018413368612527847, 0.06691651791334152, 0.0057060555554926395]], [[0.16225850582122803, -0.09203501790761948, -0.13466782867908478], [-0.5326682925224304, 0.27809661626815796, 0.45026105642318726], [0.08496968448162079, -0.054982494562864304, -0.07609446346759796], [0.15822747349739075, -0.08993607759475708, -0.1293654590845108], [-2.595245599746704, 1.316698431968689, 2.192431926727295]], [[0.12568596005439758, -0.0752398893237114, -0.11096905916929245], [0.09755513817071915, -0.06621907651424408, -0.07749399542808533], [0.04933895543217659, -0.0357004851102829, -0.038849130272865295], [0.05952027812600136, -0.04286380112171173, -0.05007566139101982], [0.1466667652130127, -0.09049273282289505, -0.12409254908561707]]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[-0.4192855274112129, -0.3612028470286608, -0.3906807018168935, -0.18680561011316651, 0.43418403461809163], [-0.48039416748463415, 0.28319083928007344, 0.3787415747147886, -0.10663818771331546, -0.2102808260027422], [0.14163438620723923, 0.3643672995613227, 0.6466593835203878, 0.26754344782969486, 0.4781527997235746]], [[-0.13508747010156097, -0.2622650951163073, -0.5159319761748147, -0.3461107760237983, -0.10903011495481445], [0.057898416625246404, -0.3939301728367635, 0.24482575917924335, -0.03091468186972795, -0.19266032253905163], [-0.22958662247510914, 0.6062064818649023, 0.05076850478631665, -0.26937796177887346, -0.2861946897121853]], [[0.551347329140022, -0.17350526896299, 0.35661122755970465, 0.42698216435196895, -0.6973204069037421], [0.4052924680488905, -0.06252963971331643, -0.3252385764805109, -0.1631527343977792, 0.16826873072021295], [0.11219296660863498, 0.12765424116862145, -0.21746880496068602, 0.3837251464090552, 0.4506391436689833]], [[0.5502191430578932, 0.3001387933393551, 0.5682943638621228, 0.6281780863270451, 0.21127679831450835], [0.26434863384995794, 0.17466273500794519, 0.1654273206327644, 0.2589733616188232, -0.5126945704582769], [-0.09713951061880531, 0.14635702547392063, -0.6321993134597211, -0.43255216480699327, 0.4753494857062574]]], "dtype": "float64"}], ["b1", {"__nd__": [[-0.10591626265092143, -0.04643473082572405, 0.03751191495613689, -0.0475981543195611, -0.08607988248871355], [-0.05160695588903406, -0.04281975203120115, -0.0579808127566781, 0.018236412585290355, 0.16098145140354342], [0.03563047678382521, 0.1624806827941641, 0.07396646728405731, 0.08360877774221787, 0.05866629023229206], [-0.029890902715660035, 0.0421640044301706, 0.03029302760458512, -0.11123286864271714, 0.07482550537168686]], "dtype": "float64"}], ["w2", {"__nd__": [[[0.1733153347832059, 0.11054851525337535, -0.16753957572157546], [-0.7900152868593612, 0.44023816182313735, -0.021090387958547233], [-0.6532246087821586, 0.1949076543410682, -0.20981926555400907], [0.20607102592511914, -0.030557954511593786, 0.340283461544743], [0.07002978681408285, 0.0838543408157203, 0.04498280736524818]], [[0.3691848138045608, -0.38877660082699256, 0.3530650690235232], [0.2066420052846708, -0.16632183814555473, -0.0007007891451986621], [-0.13395184925102563, 0.017805130180820188, 0.2821009187291959], [-0.20379074090280977, 0.22294793083390066, 0.2822077304036695], [0.19117626293291218, 0.4688649094941081, -0.021591698001178817]], [[0.20785705126439438, 0.7435578486101468, 0.00855960529773087], [-0.0417511812048884, 0.0013083757988210454, 0.1759934322881588], [-0.18546216307214475, -0.07353676770510942, -0.5264478954601624], [-0.3148436907221949, -0.3063361998005162, -0.348020682808397], [-0.1332465615365556, 0.20804015042007737, -0.1149555546005673]], [[-0.39696000106322665, 0.0726865535354096, 0.050852544771356825], [-0.4205193423685934, 0.04452587290877224, -0.32201167898642946], [-0.18973758192025633, -0.22133853823048003, 0.24606697546374598], [-0.24782205085406095, -0.09645216299519556, 0.05427830801339515], [0.30938681884505714, 0.3541006195358496, -0.3061710166364671]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.11834670515455736, 0.029742479238392308, 0.11941634398527437], [0.1052469845112465, -0.05330355135671144, 0.0038009620695226513], [-0.029476355634257802, 0.03730640789512084, 0.16532546900049772], [0.005444305932063776, -0.06639771269713039, 0.2533685004698796]], "dtype": "float64"}]]}, {"__nd__": [[-0.9941497685659371, 0.0007545597703792161, -0.014751124658677827], [-1.270336416589952, 0.7919071077140544, 1.0629464836528513], [-0.08175841953991256, 0.05747763942419942, -0.16287876735084655], [-1.7613771573612063, 0.2435098883321344, -1.243281001073121], [-0.9567565042981796, 0.3968918326546037, 0.9249180794046787], [-0.8669158826508802, 1.140159494666554, 1.7667422643376332]], "dtype": "float64"}, {"__nd__": [[0.17167969525236756, -0.075574596136044, 0.20367313281838842], [-0.3998259722154415, -0.8517494624499372, 0.646338292231823], [0.7208530221284463, -0.49751122444937135, -0.40118783306815836], [-0.6189057971264504, -0.6970104276971278, -0.9638897212512467], [-1.5014293030052674, 0.9053295359389235, -0.3470388429725984], [0.07084728905744735, 1.7912132147949598, -0.4821800858142817]], "dtype": "float64"}, "full"], "e": {"__tup__": [{"__nd__": 2.7083706855773926, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [[-0.17856240272521973, 1.5553213357925415, 1.0138877630233765, -0.07323996722698212, -0.005594316869974136], [-0.24064354598522186, -0.2902124524116516, 0.4828817844390869, 0.6110758781433105, 0.07186605036258698], [0.0298761073499918, 0.1942461133003235, 0.003975574858486652, -0.02646917849779129, 0.17292815446853638], [-0.022871771827340126, 0.049589209258556366, -0.00019163032993674278, -0.005808226298540831, -0.03664916008710861]], "dtype": "float32"}], ["b2", {"__nd__": [[-1.5840599536895752, 0.887549102306366, 0.7382519245147705], [-1.2242109775543213, 0.7016735672950745, 1.0195949077606201], [-1.0765098333358765, 0.669587254524231, 0.9245399236679077], [-1.0673084259033203, 0.6497179865837097, 0.9298481345176697]], "dtype": "float32"}], ["w1", {"__nd__": [[[0.19407159090042114, -1.7063817977905273, -1.1038379669189453, 0.09337353706359863, 0.006486077792942524], [-0.11453713476657867, 1.1506328582763672, 0.7575702667236328, -0.058085549622774124, -0.008597932755947113], [-0.1420070081949234, 1.4735954999923706, 1.0035014152526855, -0.060932550579309464, -0.013011706061661243]], [[0.5117800235748291, 0.6940218210220337, -1.1102592945098877, -1.3619694709777832, -0.09992129355669022], [-0.2449558526277542, -0.40766027569770813, 0.6307113766670227, 0.7373515367507935, 0.027657391503453255], [-0.06572357565164566, -0.2764820158481598, 0.3922184109687805, 0.38863295316696167, -0.04684901610016823]], [[0.009637018665671349, -0.4535869061946869, -0.05718676745891571, 0.01850547268986702, -0.3869594633579254], [-0.0030547287315130234, 0.25099241733551025, 0.030298756435513496, -0.012223544530570507, 0.1903073638677597], [-0.008823418989777565, 0.28170663118362427, 0.0458243153989315, -0.020338552072644234, 0.16013500094413757]], [[0.1442510187625885, -0.12846414744853973, 0.026469485834240913, 0.03421185910701752, 0.019585315138101578], [-0.09679268300533295, 0.08466088771820068, -0.014316953718662262, -0.01993240788578987, -0.00011891928443219513], [-0.08188003301620483, 0.06239943578839302, -0.016017530113458633, -0.01773044653236866, -0.00976521521806717]]], "dtype": "float32"}], ["w2", {"__nd__": [[[-0.13561083376407623, 0.08899946510791779, 0.05589160695672035], [-1.1259511709213257, 0.5256670117378235, 0.5109240412712097], [-1.8674347400665283, 0.8500869870185852, 0.8615576028823853], [-0.31977832317352295, 0.14216358959674835, 0.14908580482006073], [0.07334066182374954, -0.07131727784872055, -0.02076495625078678]], [[-0.11396612972021103, 0.08760681748390198, 0.08220042288303375], [-0.4979013502597809, 0.17605507373809814, 0.4297879636287689], [-1.669232726097107, 0.7760934233665466, 1.3490734100341797], [-0.4987509846687317, 0.3112529516220093, 0.3842129707336426], [-0.018413368612527847, 0.06691651791334152, 0.0057060555554926395]], [[0.16225850582122803, -0.09203501790761948, -0.13466782867908478], [-0.5326682925224304, 0.27809661626815796, 0.45026105642318726], [0.08496968448162079, -0.054982494562864304, -0.07609446346759796], [0.15822747349739075, -0.08993607759475708, -0.1293654590845108], [-2.595245599746704, 1.316698431968689, 2.192431926727295]], [[0.12568596005439758, -0.0752398893237114, -0.11096905916929245], [0.09755513817071915, -0.06621907651424408, -0.07749399542808533], [0.04933895543217659, -0.0357004851102829, -0.038849130272865295], [0.05952027812600136, -0.04286380112171173, -0.05007566139101982], [0.1466667652130127, -0.09049273282289505, -0.12409254908561707]]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[-0.4192855274112129, -0.3612028470286608, -0.3906807018168935, -0.18680561011316651, 0.43418403461809163], [-0.48039416748463415, 0.28319083928007344, 0.3787415747147886, -0.10663818771331546, -0.2102808260027422], [0.14163438620723923, 0.3643672995613227, 0.6466593835203878, 0.26754344782969486, 0.4781527997235746]], [[-0.13508747010156097, -0.2622650951163073, -0.5159319761748147, -0.3461107760237983, -0.10903011495481445], [0.057898416625246404, -0.3939301728367635, 0.24482575917924335, -0.03091468186972795, -0.19266032253905163], [-0.22958662247510914, 0.6062064818649023, 0.05076850478631665, -0.26937796177887346, -0.2861946897121853]], [[0.551347329140022, -0.17350526896299, 0.35661122755970465, 0.42698216435196895, -0.6973204069037421], [0.4052924680488905, -0.06252963971331643, -0.3252385764805109, -0.1631527343977792, 0.16826873072021295], [0.11219296660863498, 0.12765424116862145, -0.21746880496068602, 0.3837251464090552, 0.4506391436689833]], [[0.5502191430578932, 0.3001387933393551, 0.5682943638621228, 0.6281780863270451, 0.21127679831450835], [0.26434863384995794, 0.17466273500794519, 0.1654273206327644, 0.2589733616188232, -0.5126945704582769], [-0.09713951061880531, 0.14635702547392063, -0.6321993134597211, -0.43255216480699327, 0.4753494857062574]]], "dtype": "float64"}], ["b1", {"__nd__": [[-0.10591626265092143, -0.04643473082572405, 0.03751191495613689, -0.0475981543195611, -0.08607988248871355], [-0.05160695588903406, -0.04281975203120115, -0.0579808127566781, 0.018236412585290355, 0.16098145140354342], [0.03563047678382521, 0.1624806827941641, 0.07396646728405731, 0.08360877774221787, 0.05866629023229206], [-0.029890902715660035, 0.0421640044301706, 0.03029302760458512, -0.11123286864271714, 0.07482550537168686]], "dtype": "float64"}], ["w2", {"__nd__": [[[0.1733153347832059, 0.11054851525337535, -0.16753957572157546], [-0.7900152868593612, 0.44023816182313735, -0.021090387958547233], [-0.6532246087821586, 0.1949076543410682, -0.20981926555400907], [0.20607102592511914, -0.030557954511593786, 0.340283461544743], [0.07002978681408285, 0.0838543408157203, 0.04498280736524818]], [[0.3691848138045608, -0.38877660082699256, 0.3530650690235232], [0.2066420052846708, -0.16632183814555473, -0.0007007891451986621], [-0.13395184925102563, 0.017805130180820188, 0.2821009187291959], [-0.20379074090280977, 0.22294793083390066, 0.2822077304036695], [0.19117626293291218, 0.4688649094941081, -0.021591698001178817]], [[0.20785705126439438, 0.7435578486101468, 0.00855960529773087], [-0.0417511812048884, 0.0013083757988210454, 0.1759934322881588], [-0.18546216307214475, -0.07353676770510942, -0.5264478954601624], [-0.3148436907221949, -0.3063361998005162, -0.348020682808397], [-0.1332465615365556, 0.20804015042007737, -0.1149555546005673]], [[-0.39696000106322665, 0.0726865535354096, 0.050852544771356825], [-0.4205193423685934, 0.04452587290877224, -0.32201167898642946], [-0.18973758192025633, -0.22133853823048003, 0.24606697546374598], [-0.24782205085406095, -0.09645216299519556, 0.05427830801339515], [0.30938681884505714, 0.3541006195358496, -0.3061710166364671]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.11834670515455736, 0.029742479238392308, 0.11941634398527437], [0.1052469845112465, -0.05330355135671144, 0.0038009620695226513], [-0.029476355634257802, 0.03730640789512084, 0.16532546900049772], [0.005444305932063776, -0.06639771269713039, 0.2533685004698796]], "dtype": "float64"}]]}, {"__nd__": [[-0.9941497685659371, 0.0007545597703792161, -0.014751124658677827], [-1.270336416589952, 0.7919071077140544, 1.0629464836528513], [-0.08175841953991256, 0.05747763942419942, -0.16287876735084655], [-1.7613771573612063, 0.2435098883321344, -1.243281001073121], [-0.9567565042981796, 0.3968918326546037, 0.9249180794046787], [-0.8669158826508802, 1.140159494666554, 1.7667422643376332]], "dtype": "float64"}, {"__nd__": [[0.17167969525236756, -0.075574596136044, 0.20367313281838842], [-0.3998259722154415, -0.8517494624499372, 0.646338292231823], [0.7208530221284463, -0.49751122444937135, -0.40118783306815836], [-0.6189057971264504, -0.6970104276971278, -0.9638897212512467], [-1.5014293030052674, 0.9053295359389235, -0.3470388429725984], [0.07084728905744735, 1.7912132147949598, -0.4821800858142817]], "dtype": "float64"}, "dots"], "e": {"__tup__": [{"__nd__": 2.7083706855773926, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [[-0.17856240272521973, 1.5553213357925415, 1.0138877630233765, -0.07323996722698212, -0.005594316869974136], [-0.24064354598522186, -0.2902124524116516, 0.4828817844390869, 0.6110758781433105, 0.07186605036258698], [0.0298761073499918, 0.1942461133003235, 0.003975574858486652, -0.02646917849779129, 0.17292815446853638], [-0.022871771827340126, 0.049589209258556366, -0.00019163032993674278, -0.005808226298540831, -0.03664916008710861]], "dtype": "float32"}], ["b2", {"__nd__": [[-1.5840599536895752, 0.887549102306366, 0.7382519245147705], [-1.2242109775543213, 0.7016735672950745, 1.0195949077606201], [-1.0765098333358765, 0.669587254524231, 0.9245399236679077], [-1.0673084259033203, 0.6497179865837097, 0.9298481345176697]], "dtype": "float32"}], ["w1", {"__nd__": [[[0.19407159090042114, -1.7063817977905273, -1.1038379669189453, 0.09337353706359863, 0.006486077792942524], [-0.11453713476657867, 1.1506328582763672, 0.7575702667236328, -0.058085549622774124, -0.008597932755947113], [-0.1420070081949234, 1.4735954999923706, 1.0035014152526855, -0.060932550579309464, -0.013011706061661243]], [[0.5117800235748291, 0.6940218210220337, -1.1102592945098877, -1.3619694709777832, -0.09992129355669022], [-0.2449558526277542, -0.40766027569770813, 0.6307113766670227, 0.7373515367507935, 0.027657391503453255], [-0.06572357565164566, -0.2764820158481598, 0.3922184109687805, 0.38863295316696167, -0.04684901610016823]], [[0.009637018665671349, -0.4535869061946869, -0.05718676745891571, 0.01850547268986702, -0.3869594633579254], [-0.0030547287315130234, 0.25099241733551025, 0.030298756435513496, -0.012223544530570507, 0.1903073638677597], [-0.008823418989777565, 0.28170663118362427, 0.0458243153989315, -0.020338552072644234, 0.16013500094413757]], [[0.1442510187625885, -0.12846414744853973, 0.026469485834240913, 0.03421185910701752, 0.019585315138101578], [-0.09679268300533295, 0.08466088771820068, -0.014316953718662262, -0.01993240788578987, -0.00011891928443219513], [-0.08188003301620483, 0.06239943578839302, -0.016017530113458633, -0.01773044653236866, -0.00976521521806717]]], "dtype": "float32"}], ["w2", {"__nd__": [[[-0.13561083376407623, 0.08899946510791779, 0.05589160695672035], [-1.1259511709213257, 0.5256670117378235, 0.5109240412712097], [-1.8674347400665283, 0.8500869870185852, 0.8615576028823853], [-0.31977832317352295, 0.14216358959674835, 0.14908580482006073], [0.07334066182374954, -0.07131727784872055, -0.02076495625078678]], [[-0.11396612972021103, 0.08760681748390198, 0.08220042288303375], [-0.4979013502597809, 0.17605507373809814, 0.4297879636287689], [-1.669232726097107, 0.7760934233665466, 1.3490734100341797], [-0.4987509846687317, 0.3112529516220093, 0.3842129707336426], [-0.018413368612527847, 0.06691651791334152, 0.0057060555554926395]], [[0.16225850582122803, -0.09203501790761948, -0.13466782867908478], [-0.5326682925224304, 0.27809661626815796, 0.45026105642318726], [0.08496968448162079, -0.054982494562864304, -0.07609446346759796], [0.15822747349739075, -0.08993607759475708, -0.1293654590845108], [-2.595245599746704, 1.316698431968689, 2.192431926727295]], [[0.12568596005439758, -0.0752398893237114, -0.11096905916929245], [0.09755513817071915, -0.06621907651424408, -0.07749399542808533], [0.04933895543217659, -0.0357004851102829, -0.038849130272865295], [0.05952027812600136, -0.04286380112171173, -0.05007566139101982], [0.1466667652130127, -0.09049273282289505, -0.12409254908561707]]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[0.1732904563303718, -0.40337763860303183], [-0.18695563536533802, 0.20764416551642723]]], "dtype": "float64"}], ["b1", {"__nd__": [[-0.026096351547906783, -0.14276578647226904]], "dtype": "float64"}], ["w2", {"__nd__": [[[-0.4592235727832189, 0.09923682574141454], [-0.27958399471054807, 0.20102471993416046]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.026565298091413392, -0.03253424249963008]], "dtype": "float64"}]]}, {"__nd__": [[-0.07381036493054083, 0.7376130042608681], [0.02407887557884542, -0.9498823665424591], [0.32154280376124567, -0.3594115223384302]], "dtype": "float64"}, {"__nd__": [[-0.2381638353998516, 0.5556881483563484], [0.3154362268667343, 0.997361642586535], [-0.19254506928852966, -0.4591253289824395]], "dtype": "float64"}, "dots"], "e": {"__tup__": [{"__nd__": 0.7393105030059814, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [[-0.07371962070465088, -0.04222656413912773]], "dtype": "float32"}], ["b2", {"__nd__": [[0.16754788160324097, -0.6012285947799683]], "dtype": "float32"}], ["w1", {"__nd__": [[[-0.014995303004980087, -0.0038196933455765247], [0.02421630546450615, 0.023813335224986076]]], "dtype": "float32"}], ["w2", {"__nd__": [[[-0.0038326687645167112, -0.06109058856964111], [-0.01033649779856205, 0.08358696848154068]]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[-0.06881016583841444, 0.07279118404651122, -0.16041177104564272, -0.07262573668825556, -0.16791529683872503, -0.15443982953613497, -0.03648471975854716, 0.47375452196789286], [-0.07636037352940613, 0.049626797820964054, 0.017609786391199915, -0.035188438380911, 0.005237071767629778, 0.3832454031754527, -0.051807823029790714, 0.3133909720519074], [0.12639005571717585, -0.17061245170393985, -0.12741386776262406, -0.09056633181054462, -0.21286768929033528, 0.22402908249482342, 0.08025486443781688, 0.10299750176464668], [-0.20092820146316348, 0.09073077176378486, -0.10505992721702664, 0.4740627291608044, -0.01798116583373399, 0.05504268285620059, 0.08074942453808312, -0.092735567523321]], [[-0.07640220064249778, -0.0769507757258251, 0.1659123579524669, -0.14222150227858074, -0.03136289528615191, 0.06393367923649784, -0.07994235633564567, -0.2663795216309912], [-0.23260725407490068, -0.08141354445149744, -0.18885213052410074, 0.1870259914722624, 0.002250222671708454, -0.005968376309194199, -0.19739045351301812, 0.10099730671402532], [-0.07073525341317569, -0.18647104395204284, -0.1257226781320437, 0.46172957601192, -0.11986467866598749, -0.17039949889525244, 0.2221221762926275, 0.1416278854821175], [-0.05749359979314818, 0.2658381585553042, -0.2867361503217712, -0.09527770694676418, -0.32089227272304965, 0.2627934341429092, 0.005483766786309214, 0.4997903520448295]], [[0.08623167445350001, 0.08911327518071618, -0.2230198957746594, -0.2336587531607103, -0.09985179653510645, 0.16905703552980966, -0.20931451531064638, -0.47004500898288337], [0.15593927169226823, 0.16713113710379157, -0.18898387922680782, 0.07368996783629934, 0.20495422395608232, 0.21410750488901564, -0.3183074461340445, -0.0022388307304544303], [-0.16534431717698284, 0.17095804044227725, 0.0013012605179126605, -0.2521841072103201, 0.0022074872053742395, 0.09685785276973596, -0.3208038785133163, -0.2834207900716229], [-0.15275219793780445, -0.0807004182640856, -0.13245752531660093, -0.3792379021827112, -0.32810482039341116, -0.03057834790912994, 0.057239906836538526, -0.19635717535379765]], [[0.009630823835236413, 0.34472512937942223, -0.06289895829718849, -0.10746330549087764, -0.1584359645364058, 0.07203949112328735, 0.025712087715218754, 0.03092670981117661], [-0.09059357521014876, 0.06382108412245932, 0.06517087912950022, -0.043144520894773906, 0.21185498918658308, -0.31786856081718956, 0.12638924420565276, -0.03692865385377021], [0.15285386536742696, 0.3611372998286574, -0.00039314351292478423, -0.04674546598498146, -0.012836900493387005, 0.2630363880056645, 0.2946769707571985, 0.026166634035833903], [0.12082144406886004, 0.34719105427680286, 0.10125230689451506, -0.29775552361735386, -0.1632871082106746, -0.09760878370930437, 0.07500942951253012, -0.1120780202407573]], [[0.27162989425596284, -0.09387267288894258, 0.30815879844774674, -0.12842729716267162, 0.26553283589914556, 0.03884959184567585, 0.2819630592348156, 0.3884878147490125], [0.34810750666779733, 0.20959472501631793, -0.14207440290100723, -0.15413752140899745, -0.6202159763149382, 0.2969450946855391, 0.25685846648471905, 0.20956683420481934], [-0.21497622677746092, 0.5533099170550573, -0.12295850585076121, 0.1814141042465673, -0.30929575885539157, -0.11009522817124012, 0.25393848387951207, 0.21541150580365587], [-0.20212646777318502, -0.004880683781462308, 0.23380742747218042, -0.3523703221109384, 0.163757018004971, -0.2953176301536089, 0.1361059776974217, -0.04061160593290068]], [[0.20514958158493968, -0.09278783612444476, -0.06141669731246862, -0.06303927669391743, -0.18374330696993502, -0.05079895453233962, -0.19809112411628257, -0.10905762229928295], [0.1371726538242016, -0.06275429732378546, 0.21420913505343867, 0.14152835461704497, 0.1654877988796208, -0.22536532826554231, 0.08309046320586033, 0.2869119091896752], [0.18093336938875598, 0.24468520458187976, 0.30295117060580357, -0.29958676296151376, 0.11431442316717566, 0.11007977565938105, -0.03778068087988242, -0.2596982450299131], [0.02994772332369135, -0.030851686622433518, 0.3889770668689474, 0.2615300803612934, -0.0805609830183742, -0.2087914170059119, -0.16762491552170827, -0.009245205417554726]], [[-0.14306228959984454, 0.3942493827750997, -0.235802142919883, -0.016299049150073238, 0.03070788570955048, 0.0846463960238586, -0.1820201542017124, -0.04220605137727357], [0.03894399872393107, 0.06842941993503784, -0.11239331938895741, -0.0570171214484824, 0.04759994440531598, 0.38463351264560974, -0.2345183200540292, 0.14368828918098092], [0.3197218736227858, -0.2708426588644119, 0.2945576937142424, 0.28757243977467933, 0.02161072852161736, 0.20086425982254116, 0.3046053234532129, -0.018591627043758347], [0.05757402181272995, -0.19980876557149693, -0.1237601737503381, 0.02812494677314575, -0.003112935674863204, 0.3356808340824465, -0.028084468005614316, -0.22397188446640556]], [[0.19278388892177623, 0.19986865163077835, 0.09601663759779601, -0.009751084140808093, -0.13900186478043558, -0.11075560322949916, 0.32001702909796526, 0.1730143662606877], [0.03850201440173834, 0.10123309995872048, -0.05043679163756326, -0.16549482134410587, -0.07597410224118722, 0.006601327361841082, -0.30907631605824254, -0.27048601514510706], [-0.32851915762304124, -0.07607811540316532, 0.1333130709230682, -0.25460868236361095, -0.21712965270387855, -0.31084462883026426, -0.08618989956908185, -0.2312009349608881], [-0.19608293998921822, -0.21587238885573048, 0.44891483795549836, -0.016700256230431962, -0.12887322555753586, -0.08888252330682848, 0.17318319204593857, 0.19379148894178752]]], "dtype": "float64"}], ["b1", {"__nd__": [[0.026890139171886734, -0.19476577401877215, 0.09802215423460969, 0.10637817323130282, 0.07748412013923961, 0.04757149157749407, 0.008252523424014151, 0.11975131425917965], [-0.029267122565772005, 0.10440220469033026, -0.046560167924185276, -0.036973570716635894, -0.04286711917979204, -0.054123576644850506, -0.047489033950691255, 0.07609647753682237], [0.15576980512140975, -0.0688962581459956, -0.10225378096921862, 0.12493309849471296, -0.03223914343803996, 0.028325795107740845, 0.01590154532245998, 0.09096617626761935], [-0.08687411277147672, 0.11426517048886792, 0.07986574508050805, -0.21743996001474294, -0.015376276300892742, -0.03727059963655181, 0.03816464083813881, -0.005861430233519479], [-0.07430703428206487, -0.034177155394694264, 0.03475619165639082, 0.00416201235983734, -0.015359643176601302, 0.21996870709617272, 0.05258566359262428, -0.11066564008740212], [0.011826249094033098, 0.03234165336727698, -0.09820540584345483, 0.011416633455922944, -0.08339429124812854, -0.22027583556484698, 0.03410752452516796, 0.06237040380880163], [0.06133361850698746, -0.01439807694635057, -0.05849194071873995, 0.023566772145516356, 0.04259751728898517, 0.09118464718644066, 0.13301376481261923, 0.0805481199088044], [-0.05664146245944577, -0.004690578129040502, -0.1382362881154112, 0.07191284700826864, 0.058480363939513794, 0.1189805261410762, 0.049575212770777005, 0.15685257071840344]], "dtype": "float64"}], ["w2", {"__nd__": [[[-0.1346446987250294, 0.06875852552780243, -0.04596292629290362, -0.04051682116171971], [-0.3473783850695885, -0.14544817065485047, -0.3921736297688012, -0.06471894589228476], [-0.1377092268436719, 0.025693421930636947, -0.3616879742772746, -0.10517324297234017], [-0.0532570272940066, -0.45973280523919735, 0.3186767731859551, 0.08947448841338593], [-0.15760455889492633, 0.36609831493358486, 0.26038282113219097, -0.4963646562628742], [-0.06662504044127214, -0.13126247063597232, 0.0007288607332563631, -0.19629698610373605], [-0.0599151886707989, 0.0068221570378235305, -0.5030013511692514, 0.015363501325126505], [0.26944846348364554, 0.016219122420457307, 0.21963226543412762, 0.30653001638630795]], [[0.050589753543924844, 0.07165224235495564, 0.07505494233161879, -0.0886419225842125], [0.4469876543011223, 0.08506735541814192, 0.10880104577782444, 0.18740160569539774], [0.292038831637939, -0.2707381751403252, -0.204236759850907, 0.027434030674410304], [0.10272480536037071, -0.24099560957708835, -0.2733136734623248, 0.15151787690119325], [0.14178170212661892, -0.05909943025514383, -0.060214695446959836, -0.12971288901299907], [-0.2378897775019247, -0.17970135523407563, 0.02675259508040518, -0.04414164896484285], [0.18379169687115826, -0.2712784983270782, -0.347254595839296, 0.1024285687251758], [-0.19830199462534862, -0.07460174793194686, -0.007065985380121239, -0.2636408743794302]], [[-0.3190883036057574, 0.03826860055811681, -0.5554517093335879, 0.10692891036086952], [-0.37018806867319065, 0.1424181446971214, 0.04380702859906888, 0.02867350904257714], [0.07970954610461617, 0.010075706041934286, -0.29025972682999507, 0.14658102058144984], [0.0084547991213395, 0.07532409330385259, -0.04265973336750547, -0.06796862083222603], [0.16683961392769908, 0.17053377519948704, 0.030414356588829808, -0.2546695264127659], [-0.20092599480255174, -0.2151245278090834, 0.2081297494792557, 0.08812683583813188], [0.002734661882037019, -0.03853446263141663, -0.1547471333136671, 0.15454700062337717], [-0.21309961630192992, -0.17423466836028906, 0.1629315123676188, 0.12142349606734518]], [[-0.016300707926102113, -0.2534851101925185, -0.25260290092237836, 0.16753711486169798], [0.07461003319397001, 0.29617437338596614, -0.09161206760453448, -0.013662918010637593], [-0.18428918195569546, 0.3312868271077107, -0.3313329123212716, 0.12627758055556768], [0.324366062718669, -0.18594109893060087, -0.2822689392109092, -0.19706905579509526], [-0.2794808601598761, 0.0726311256986568, -0.015135074279329681, -0.17746431517590983], [-0.05286007187864475, 0.3038174169466925, -0.04674442093756677, -0.07574394950391568], [0.3899339959032726, -0.17410575333704123, -0.09003708101167342, 0.295366222238456], [0.009176716170333062, -0.3787946239690414, -0.1724296923750832, -0.4491167681838566]], [[-0.13039912508438822, 0.06178966646908152, -0.33074226327519946, 0.30252409875845077], [0.24820640539713798, 0.15621375640921928, -0.14998647502834134, 0.001839191658190411], [0.10658776833221223, 0.18736559698979108, 0.2514075914711478, -0.4357441877412682], [-0.14628919714920102, -0.23606960413739975, -0.07216634562465278, 0.037389129819498906], [0.4545462262612239, -0.02363284139357344, 0.17675140193309977, 0.017665432652669448], [-0.011599942416971078, 0.34486713775623506, -0.22709041026558424, 0.13065676511129928], [0.4106524770188168, 0.6815494859633634, -0.10710520424571818, -0.1904969068725234], [-0.256255971072431, -0.08649107358594814, -0.0045585903972810695, -0.0017034244328057381]], [[0.22140216937187276, -0.06795444027253508, 0.05881452068147141, -0.33715266089530416], [-0.13862165277819724, -0.03695012331486865, -0.4809171421075717, 0.017535010385672744], [0.1704381809498055, 0.0090665812881524, 0.03699312691967906, -0.007400288110601532], [-0.24608647449719234, 0.002663081815795915, -0.11923438032901817, 0.012184901188399183], [0.007777300412755148, 0.017358659603431838, -0.2712069208244606, -0.01847155115419088], [0.3442658787487654, 0.18130328415529462, -0.05347015872435565, -0.15360988991295255], [0.191192676691421, -0.008225459874193463, -0.3644558145475188, -0.09834143644956622], [0.3288639215710646, -0.3488453551337905, 0.002341222294049639, -0.16248339311574175]], [[-0.024936332560448655, -0.09556561897858436, -0.1897275185534042, -0.5310242034381097], [-0.057786099599627654, -0.008505835383619395, -0.04236832455051565, -0.22998103160299477], [-0.13983094232129142, 0.4295792669382107, -0.24282904746375206, -0.08259770065686262], [-0.2681762576497601, -0.06196485332836287, 0.03305611344961111, 0.22743706898051516], [0.028478940051032353, -0.015305072586214857, 0.057074538532904645, 0.098333303828098], [-0.022264905409977626, -0.019031967973725486, 0.23189671026051686, -0.1583838940764243], [-0.03135187826600162, -0.19970868848005083, -0.11887527874237061, 0.28416651900568013], [-0.15701612514006738, 0.08747263127392518, 0.40931007096554756, 0.09010141443475041]], [[-0.04780527137866286, -0.17205667402402638, -0.5275250885635429, -0.2082227105072728], [-0.20256464106357241, 0.17543193928377718, 0.12257483948376709, -0.2784069369146593], [0.10685910806043786, -0.1463261299184306, 0.3492597042967559, 0.2801858653407276], [0.11897928209716675, -0.0939122182991296, 0.3424145699029997, 0.16540911022290417], [0.22370971754028512, 0.10161814824716871, -0.05651135521118459, 0.15577343202121333], [-0.16834069821996897, 0.1587013125380653, 0.0362791828865368, -0.33279860573530806], [0.09690722399211038, -0.011647770371008467, -0.07947382520424015, -0.01889203802912143], [0.1542694733997332, -0.23105830775880967, -0.21841894111198898, 0.1434142077290315]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.0165938633491487, -0.17591626341404779, 0.009474709626156881, -0.1735974072924747], [0.06674972070754114, 0.024627141487189308, 0.044858838696378155, -0.15829460282696717], [-0.0843240611554156, 0.08067510321684208, -0.14696189059744522, 0.045157138938493685], [-0.029046433624649606, 0.007231372096326275, -0.09069075268068988, 0.052267445652637745], [0.03194452771842115, 0.07801187237951922, 0.24158894079789128, -0.028287074860047886], [0.0009141035812949247, -0.11149949969531031, 0.16986367040146436, -0.04062952706546058], [0.07284555067442676, -0.11834852515987904, -0.029202579752653713, 0.0569926812038542], [0.04497725460886387, -0.030225232604986385, 0.08827208832314004, 0.010375581881747977]], "dtype": "float64"}]]}, {"__nd__": [[1.0530347472411112, -0.2389504645349172, 0.6560946740329053, -0.46442069578093315], [1.2918822472047888, 0.01738898118751436, 0.07599590387443536, -0.5532064889206864]], "dtype": "float64"}, {"__nd__": [[1.0375259734623878, 0.18264476166027943, -0.19939071921693807, 1.3493402726955501], [-0.7410473320327328, 1.009171554785022, 1.525386204813374, -1.7096234809718305]], "dtype": "float64"}, "full"], "e": {"__tup__": [{"__nd__": 1.7191503047943115, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [[-0.06970755010843277, -0.034435488283634186, -0.02534332126379013, 0.05089383199810982, -0.10064738988876343, 0.02800709195435047, -0.013417325913906097, 0.1072554960846901], [0.004702158272266388, 0.08644892275333405, 0.2634306848049164, 0.05974065884947777, 0.07462677359580994, -0.030695840716362, 0.10368722677230835, -0.0020907819271087646], [-0.14741875231266022, -0.21439535915851593, 0.002583194524049759, -0.006182004697620869, 0.038325902074575424, -0.06609229743480682, 0.000985797494649887, -0.005894320085644722], [0.02605198323726654, -0.051096297800540924, -0.11981827765703201, 0.12546098232269287, -0.08400781452655792, -0.09967491775751114, 0.18389417231082916, 0.13477715849876404], [-0.08722002804279327, 0.05885930731892586, 0.07662217319011688, -0.03347264230251312, 0.2135860025882721, -0.0834905207157135, 0.14624303579330444, -0.13918109238147736], [0.20346787571907043, -0.04459182545542717, 0.0363682322204113, -0.0480077750980854, 0.005867229774594307, 0.077546626329422, 0.07385408133268356, 0.1740657091140747], [0.10622870922088623, 0.008862853050231934, -0.07279399037361145, -0.1426474153995514, -0.0013888999819755554, 0.01692119985818863, -0.04631362855434418, -0.11839324980974197], [0.06037822365760803, -0.09712859988212585, 0.0074327122420072556, 0.024085339158773422, 0.030803896486759186, -0.03963779658079147, 0.05558115988969803, 0.08117033541202545]], "dtype": "float32"}], ["b2", {"__nd__": [[0.7259061336517334, -0.5431342720985413, -0.07884106040000916, -0.20881974697113037], [0.7097427845001221, -0.4760991036891937, -0.07587361335754395, -0.1178135871887207], [0.7531281113624573, -0.4097384810447693, -0.060195595026016235, -0.15029612183570862], [0.7614115476608276, -0.4230468273162842, -0.07248362898826599, -0.1083497703075409], [0.7220823168754578, -0.25042933225631714, -0.05859372019767761, -0.240590900182724], [0.714040994644165, -0.3217065632343292, -0.07110938429832458, -0.21899819374084473], [0.6914375424385071, -0.34305691719055176, -0.0316716730594635, -0.2618422508239746], [0.6667928695678711, -0.2894518971443176, 0.0038402676582336426, -0.2988141179084778]], "dtype": "float32"}], ["w1", {"__nd__": [[[-0.08478210121393204, -0.03478368744254112, -0.02438276633620262, 0.060464780777692795, -0.143907830119133, 0.028861690312623978, -0.0001457735343137756, 0.1381417214870453], [0.004445785190910101, 0.00981470663100481, 0.008529209531843662, -0.004786044359207153, -0.01665014587342739, -0.007369252387434244, 0.018213264644145966, 0.0014145741006359458], [-0.018101437017321587, -0.02618282288312912, -0.022224972024559975, 0.016701335087418556, 0.026070039719343185, 0.019907236099243164, -0.04276445508003235, 0.009170440956950188], [0.036602988839149475, 0.015443112701177597, 0.010913270525634289, -0.02619057707488537, 0.06083955615758896, -0.012772606685757637, 0.00103339075576514, -0.05917840451002121]], [[-0.0002515800588298589, 0.12628981471061707, 0.39713054895401, 0.10567650943994522, 0.10473465174436569, -0.04502911865711212, 0.1692529320716858, -0.00609265873208642], [-0.007909685373306274, -0.01450488530099392, -0.03190487250685692, 0.008380007930099964, -0.016805924475193024, 0.004963543266057968, 0.00038302448228932917, -0.0026874954346567392], [0.016258161514997482, 0.03102385438978672, 0.06937866657972336, -0.01621910184621811, 0.035548608750104904, -0.010633601807057858, 0.000827551179099828, 0.005466760601848364], [-0.0006841901922598481, -0.041366323828697205, -0.12875677645206451, -0.03263314440846443, -0.034767184406518936, 0.014729213900864124, -0.05352475866675377, 0.0016685613663867116]], [[-0.19718037545681, -0.3365197777748108, 0.013121014460921288, -0.011366955935955048, 0.04674956202507019, -0.10549019277095795, 0.00647474592551589, -0.009188540279865265], [0.06772780418395996, 0.041613928973674774, 0.0098641999065876, -0.0007020084303803742, -0.022767983376979828, 0.010827403515577316, 0.005442177411168814, 0.0012164735235273838], [-0.12357762455940247, -0.08590353280305862, -0.016060810536146164, 0.0006598326144739985, 0.04063820838928223, -0.02318144030869007, -0.008896301500499249, -0.0024811201728880405], [0.07791643589735031, 0.1287326067686081, -0.004360280465334654, 0.004227401223033667, -0.01885823905467987, 0.04022709280252457, -0.0021186750382184982, 0.0035195956006646156]], [[0.050829607993364334, -0.07034744322299957, -0.15322308242321014, 0.17536193132400513, -0.12022918462753296, -0.14135152101516724, 0.2688576579093933, 0.17754653096199036], [0.01752524822950363, 0.005810982082039118, 0.029699750244617462, -0.010664118453860283, 0.003295533824712038, 0.005690102931112051, 0.0005561603466048837, -0.026295309886336327], [-0.030566703528165817, -0.012235027737915516, -0.05756469815969467, 0.023567305877804756, -0.008873163722455502, -0.013725515455007553, 0.005465155933052301, 0.05197476968169212], [-0.02016211859881878, 0.029023751616477966, 0.06385103613138199, -0.07220801711082458, 0.04935669153928757, 0.05809548497200012, -0.11007674783468246, -0.07368472963571548]], [[-0.09275443851947784, 0.08324567973613739, 0.06608488410711288, -0.03905288130044937, 0.29358500242233276, -0.10571315139532089, 0.18080554902553558, -0.19235548377037048], [0.036393992602825165, 0.002833103993907571, -0.05239810049533844, 0.009382449090480804, -0.000985057675279677, 0.012387903407216072, -0.027486002072691917, -0.0007427201489917934], [-0.08718851208686829, 0.0037606977857649326, 0.11766402423381805, -0.0242427010089159, 0.0362977609038353, -0.038321997970342636, 0.07876117527484894, -0.020869100466370583], [0.03142353519797325, -0.03181646391749382, -0.019693268463015556, 0.013835313729941845, -0.11109039187431335, 0.03877582028508186, -0.06567854434251785, 0.07292723655700684]], [[0.28316834568977356, -0.05544780194759369, 0.053238898515701294, -0.07093031704425812, 0.011853531002998352, 0.11092433333396912, 0.1130782887339592, 0.2589605450630188], [0.04197195917367935, 0.004432962741702795, 0.012914358638226986, -0.01839289255440235, 0.008814728818833828, 0.02218617871403694, 0.03646177798509598, 0.07036442309617996], [0.12341459840536118, -0.052582208067178726, 0.011921098455786705, -0.013216309249401093, -0.010685943067073822, 0.0354418121278286, 0.0050338986329734325, 0.04103383421897888], [-0.11312641203403473, 0.023079240694642067, -0.020900726318359375, 0.027759045362472534, -0.004217981360852718, -0.04389328137040138, -0.04373040050268173, -0.10111026465892792]], [[0.1426817774772644, 0.00040496530709788203, -0.10821471363306046, -0.20566150546073914, 0.0008435816271230578, 0.011470971629023552, -0.054403942078351974, -0.1867333948612213], [-0.0029675252735614777, -0.020770562812685966, -0.01660079136490822, -0.021114949136972427, 0.00487378379330039, -0.020563090220093727, 0.015219125896692276, -0.04615228995680809], [0.1138516366481781, 0.05053092911839485, -0.04076162725687027, -0.10270102322101593, -0.01115526631474495, 0.05830266326665878, -0.07747834920883179, -0.028003260493278503], [-0.06830710172653198, -0.0027179261669516563, 0.04951464757323265, 0.09537088871002197, 0.00019078607147093862, -0.007962437346577644, 0.027757829055190086, 0.08331336826086044]], [[0.09821311384439468, -0.17228801548480988, 0.013215664774179459, 0.03578237071633339, 0.05300280079245567, -0.07038390636444092, 0.08782391995191574, 0.13819773495197296], [0.015495172701776028, -0.04683414101600647, 0.0036321058869361877, 0.0009775019716471434, 0.012343721464276314, -0.01922600157558918, 0.010300903581082821, 0.030276549980044365], [0.017540330067276955, 0.00997810810804367, -0.0008475150680169463, 0.016069265082478523, 0.0012107035145163536, 0.004269338212907314, 0.02305634319782257, 0.0071130236610770226], [-0.045466676354408264, 0.08003160357475281, -0.006139518693089485, -0.016500264406204224, -0.02459232322871685, 0.03269617632031441, -0.040607769042253494, -0.06409471482038498]]], "dtype": "float32"}], ["w2", {"__nd__": [[[0.035229723900556564, -0.0210834089666605, 0.010638853535056114, -0.026824073866009712], [-0.06142415478825569, 0.042063262313604355, -0.004008221440017223, 0.029991570860147476], [-0.026279423385858536, 0.01754520647227764, -0.002951204776763916, 0.014257924631237984], [-0.07317757606506348, 0.05523408204317093, 0.009267927147448063, 0.019527750089764595], [-0.05437246710062027, 0.037677690386772156, -0.0023323316127061844, 0.02514575608074665], [-0.04286397993564606, 0.03500794246792793, 0.012706282548606396, 0.003041730262339115], [-0.017728516831994057, 0.016774198040366173, 0.011547296307981014, -0.00600150553509593], [0.4195811152458191, -0.327828586101532, -0.08365699648857117, -0.07675701379776001]], [[-0.031148836016654968, 0.022416729480028152, 0.008788679726421833, -0.0007664631702937186], [-0.05795851722359657, 0.03755868226289749, 0.0014607716584578156, 0.0147708086296916], [0.1376386135816574, -0.09741582721471786, -0.03296026214957237, -0.003002489684149623], [-0.009954838082194328, 0.016904201358556747, 0.03774382919073105, -0.03824058547616005], [0.005076956003904343, -0.006766852457076311, -0.012598516419529915, 0.012269202619791031], [-0.04816921800374985, 0.02933371067047119, -0.005533486604690552, 0.01961466111242771], [-0.008848830126225948, 0.014471328817307949, 0.031560566276311874, -0.031827811151742935], [-0.10853557288646698, 0.07474177330732346, 0.0185451190918684, 0.010465791448950768]], [[0.1302000731229782, -0.08024392277002335, -0.04315433278679848, 0.013256951235234737], [0.06587570160627365, -0.03560853376984596, -0.004461041186004877, -0.014109990559518337], [-0.09169740974903107, 0.05210493132472038, 0.015045790933072567, 0.009052885696291924], [-0.04423810541629791, 0.02042677253484726, -0.009136646054685116, 0.024013062939047813], [-0.011967373080551624, 0.004176427144557238, -0.007168561220169067, 0.01212411466985941], [0.1435750275850296, -0.08190559595823288, -0.024680085480213165, -0.012829907238483429], [-0.10582880675792694, 0.057519569993019104, 0.008262073621153831, 0.021354978904128075], [-0.12596255540847778, 0.06839971244335175, 0.009615144692361355, 0.02567986585199833]], [[-0.031709831207990646, 0.02230016700923443, 0.01915295049548149, -0.012038066983222961], [0.2470064014196396, -0.12726031243801117, 0.010872980579733849, -0.07042329013347626], [-0.02573746256530285, 0.014379329979419708, 0.0027236398309469223, 0.0033818904776126146], [-0.06479786336421967, 0.035629551857709885, 0.0048841568641364574, 0.010538304224610329], [-0.05556352064013481, 0.029277777299284935, -0.00020291854161769152, 0.013540774583816528], [0.10208562016487122, -0.043171390891075134, 0.03697037696838379, -0.06241961941123009], [0.03772137314081192, -0.014504180289804935, 0.018650617450475693, -0.028182988986372948], [0.04603119194507599, -0.025060763582587242, -0.0026087695732712746, -0.00836926233023405]], [[0.19029030203819275, -0.09321817755699158, -0.0722241997718811, 0.029843367636203766], [-0.02831767499446869, 0.025338025763630867, 0.034664496779441833, -0.04371575266122818], [0.14654654264450073, -0.05701249837875366, -0.024798788130283356, -0.02763233706355095], [0.011549354530870914, 0.004869544878602028, 0.0175750944763422, -0.03424808382987976], [0.09264936298131943, -0.03525754064321518, -0.014037095010280609, -0.020164666697382927], [0.2054491639137268, -0.08905664831399918, -0.053807664662599564, -0.0074701192788779736], [0.19440124928951263, -0.06483414024114609, -0.010378146544098854, -0.07363472878932953], [0.2387411892414093, -0.08526219427585602, -0.02451040968298912, -0.07110948860645294]], [[0.18541774153709412, -0.08079040050506592, -0.011598136276006699, -0.06925728917121887], [-0.019176499918103218, 0.01751605048775673, 0.024087995290756226, -0.03413114696741104], [-0.07330908626317978, 0.03754052519798279, 0.018573405221104622, 0.002146485261619091], [-0.06835200637578964, 0.02525963820517063, -0.007025160361081362, 0.04591882601380348], [-0.06414864212274551, 0.030287785455584526, 0.00985154788941145, 0.013426537625491619], [-0.06945764273405075, 0.03596963360905647, 0.018600497394800186, 0.00022437407460529357], [-0.043798770755529404, 0.018757060170173645, 0.0019226624863222241, 0.017833733931183815], [-0.009501278400421143, -0.005530137103050947, -0.023567449301481247, 0.04714006930589676]], [[-0.013282758183777332, 0.006682869512587786, 0.027715345844626427, -0.043222248554229736], [0.3164926767349243, -0.15722675621509552, -0.07268261909484863, -0.016278916969895363], [-0.06433023512363434, 0.03199730068445206, 0.026311179623007774, -0.01722916215658188], [0.034416571259498596, -0.016971943899989128, 0.028814204037189484, -0.06713095307350159], [0.043242197483778, -0.021458003669977188, -0.0029686426278203726, -0.014616945758461952], [0.07460065186023712, -0.03701004758477211, -0.002514259424060583, -0.029857873916625977], [-0.01667673885822296, 0.008402758277952671, 0.038395799696445465, -0.06067224219441414], [0.08185088634490967, -0.04064219444990158, -0.013071976602077484, -0.014401128515601158]], [[0.088823601603508, -0.033860690891742706, -0.03734768182039261, 0.023002061992883682], [0.19406373798847198, -0.0809418112039566, -0.02548319473862648, -0.04283710941672325], [-0.06676369905471802, 0.027835605666041374, 0.008853930048644543, 0.014592988416552544], [-0.025147903710603714, 0.012015917338430882, -0.009005236439406872, 0.025968780741095543], [-0.052144166082143784, 0.023448510095477104, -0.006852644495666027, 0.03423777222633362], [-0.04183697700500488, 0.019842278212308884, -0.013789849355816841, 0.04122580587863922], [0.14143957197666168, -0.061430685222148895, 0.0010755830444395542, -0.06381717324256897], [0.049327995628118515, -0.020367052406072617, -0.008146661333739758, -0.008119338192045689]]], "dtype": "float32"}]]}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(deep_mlp_loss_and_grads, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")